# MVP de Machine Learning & Analytics

**Nome:** Lívia Brandão  
**Matrícula:** 4052025002389  
**Dataset:** Most Streamed Spotify Songs 2023  
**Tipo de problema:** Classificação supervisionada

## 1. Apresentação do problema

O consumo de música por streaming transformou a forma como o sucesso de uma faixa é medido. Plataformas como o Spotify disponibilizam dados que permitem observar padrões de desempenho, presença em playlists, aparição em rankings e características musicais das faixas.

Neste MVP, o objetivo é construir um modelo de Machine Learning capaz de classificar se uma música pertence ao grupo de maior sucesso da base, considerando como referência o número de streams. Para isso, foi criada uma variável-alvo binária chamada `sucesso`, em que músicas com número de streams acima da mediana da base foram classificadas como `1` e as demais como `0`.

Esse problema pode ser tratado como um problema de classificação supervisionada, pois existe uma variável-alvo definida e um conjunto de atributos explicativos que podem ser utilizados para treinar modelos capazes de aprender padrões associados ao desempenho das músicas.

### Hipóteses consideradas

1. Músicas com maior presença em playlists e charts tendem a apresentar maior probabilidade de sucesso.
2. Características musicais como energia, dançabilidade e valência podem contribuir para diferenciar faixas de maior e menor desempenho.
3. Variáveis de exposição parecem ter maior influência sobre o sucesso do que atributos musicais isolados.
4. O uso de modelos de Machine Learning pode superar um baseline simples baseado em regra ingênua.

## 2. Importação das bibliotecas

Nesta etapa são importadas as bibliotecas utilizadas para leitura dos dados, visualização, preparação, modelagem, otimização e avaliação dos resultados.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 3. Carga dos dados

O dataset deve ser carregado diretamente de uma URL pública do GitHub, em formato Raw. Para a entrega, o arquivo `spotify-2023.csv` deve estar no mesmo repositório público do GitHub em que o notebook será disponibilizado.

**Atenção:** antes da entrega, substitua o valor de `DATASET_URL` pela URL Raw do seu arquivo no GitHub.

In [ ]:
# Substitua pela URL Raw do arquivo spotify-2023.csv no seu repositório do GitHub.
# Exemplo de formato:
# DATASET_URL = 'https://raw.githubusercontent.com/SEU_USUARIO/SEU_REPOSITORIO/main/spotify-2023.csv'

DATASET_URL = 'COLE_AQUI_A_URL_RAW_DO_SEU_CSV_NO_GITHUB'

if DATASET_URL == 'COLE_AQUI_A_URL_RAW_DO_SEU_CSV_NO_GITHUB':
    raise ValueError(
        'Substitua DATASET_URL pela URL Raw do arquivo spotify-2023.csv no GitHub antes de executar o notebook.'
    )

df = pd.read_csv(DATASET_URL, encoding='latin1')
df.head()

## 4. Apresentação dos dados

A base escolhida contém informações sobre músicas populares no Spotify em 2023, incluindo dados de identificação da faixa, artista, data de lançamento, número de streams, presença em playlists e charts, além de atributos musicais.

As principais variáveis utilizadas no projeto são:

- `track_name`: nome da música;
- `artist(s)_name`: artista ou artistas da faixa;
- `streams`: número total de reproduções;
- `in_spotify_playlists`: quantidade de playlists do Spotify em que a música aparece;
- `in_spotify_charts`: presença da música em rankings do Spotify;
- `bpm`: batidas por minuto;
- `danceability_%`: grau de dançabilidade;
- `energy_%`: nível de energia da música;
- `valence_%`: positividade percebida da faixa;
- `acousticness_%`, `instrumentalness_%`, `liveness_%` e `speechiness_%`: outros atributos musicais.

A variável-alvo será criada a partir de `streams`.

In [ ]:
print(f'Quantidade de registros: {df.shape[0]}')
print(f'Quantidade de atributos: {df.shape[1]}')

df.info()

In [ ]:
df.describe(include='all').T

## 5. Análise exploratória inicial

A análise exploratória tem como objetivo compreender a estrutura da base antes da modelagem. Nesta etapa são avaliados tipos de dados, valores ausentes, distribuição da variável principal e relações iniciais entre variáveis relevantes.

In [ ]:
# Conversão de colunas numéricas que podem ter sido importadas como texto

colunas_possivelmente_texto = [
    'streams',
    'in_deezer_playlists',
    'in_shazam_charts'
]

for coluna in colunas_possivelmente_texto:
    if coluna in df.columns:
        df[coluna] = (
            df[coluna]
            .astype(str)
            .str.replace(',', '', regex=False)
            .replace('nan', np.nan)
        )
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

df.dtypes

In [ ]:
# Valores ausentes por coluna

valores_ausentes = df.isnull().sum().sort_values(ascending=False)
valores_ausentes[valores_ausentes > 0]

### Distribuição de streams

O número de streams é a referência usada para construir a variável-alvo do problema. A distribuição dessa variável ajuda a entender se o sucesso está bem distribuído ou concentrado em poucas músicas.

In [ ]:
sns.histplot(df['streams'], kde=True)
plt.title('Distribuição do número de streams')
plt.xlabel('Streams')
plt.ylabel('Frequência')
plt.show()

In [ ]:
sns.boxplot(x=df['streams'])
plt.title('Boxplot do número de streams')
plt.xlabel('Streams')
plt.show()

A distribuição de streams tende a apresentar forte assimetria, com poucas músicas concentrando volumes muito altos de reprodução. Esse comportamento é comum em plataformas digitais e reforça a decisão de transformar o problema em uma classificação entre músicas acima e abaixo da mediana de streams.

### Criação da variável-alvo

A variável `sucesso` foi criada com base na mediana de streams. Essa escolha divide a base em dois grupos de tamanho semelhante, reduzindo o risco de desbalanceamento severo entre as classes.

In [ ]:
mediana_streams = df['streams'].median()

df['sucesso'] = np.where(df['streams'] > mediana_streams, 1, 0)

print(f'Mediana de streams: {mediana_streams:,.0f}')
df['sucesso'].value_counts(normalize=True).rename('proporcao')

In [ ]:
sns.countplot(data=df, x='sucesso')
plt.title('Distribuição da variável-alvo')
plt.xlabel('Sucesso (0 = abaixo ou igual à mediana, 1 = acima da mediana)')
plt.ylabel('Quantidade de músicas')
plt.show()

A variável-alvo ficou praticamente balanceada, pois foi construída a partir da mediana. Isso facilita o treinamento e a avaliação dos modelos, já que as métricas não ficam excessivamente influenciadas por uma classe majoritária.

### Relação entre exposição e sucesso

A seguir, são observadas variáveis ligadas à exposição da música em playlists e charts.

In [ ]:
variaveis_exposicao = [
    'in_spotify_playlists',
    'in_spotify_charts',
    'in_apple_playlists',
    'in_apple_charts',
    'in_deezer_playlists',
    'in_deezer_charts',
    'in_shazam_charts'
]

variaveis_exposicao = [v for v in variaveis_exposicao if v in df.columns]

df.groupby('sucesso')[variaveis_exposicao].mean().T

In [ ]:
sns.boxplot(data=df, x='sucesso', y='in_spotify_playlists')
plt.title('Presença em playlists do Spotify por classe de sucesso')
plt.xlabel('Sucesso')
plt.ylabel('Quantidade de playlists')
plt.show()

A comparação entre grupos sugere que músicas classificadas como sucesso tendem a apresentar maior presença em playlists. Esse resultado sustenta a hipótese de que variáveis de exposição têm papel importante no desempenho das faixas.

### Correlação entre variáveis numéricas

A matriz de correlação permite observar associações lineares entre os atributos numéricos.

In [ ]:
df_numerico = df.select_dtypes(include=np.number)

plt.figure(figsize=(14, 10))
sns.heatmap(df_numerico.corr(), cmap='coolwarm', center=0)
plt.title('Matriz de correlação das variáveis numéricas')
plt.show()

## 6. Preparação dos dados

Nesta etapa são selecionados os atributos utilizados na modelagem e são definidas as transformações necessárias. Para evitar vazamento de dados, as transformações serão aplicadas dentro de pipelines, permitindo que imputação, padronização e codificação sejam ajustadas apenas nos dados de treino.

In [ ]:
# Cópia de trabalho
dados_modelo = df.copy()

# A variável streams foi usada para criar o alvo. Portanto, ela não será usada como atributo preditor,
# evitando vazamento de informação.
colunas_remover = [
    'streams',
    'sucesso',
    'track_name'
]

# Algumas colunas textuais de alta cardinalidade ou de identificação são removidas para simplificar o MVP.
colunas_remover += [
    'artist(s)_name'
]

colunas_remover = [col for col in colunas_remover if col in dados_modelo.columns]

X = dados_modelo.drop(columns=colunas_remover)
y = dados_modelo['sucesso']

# Separação de atributos numéricos e categóricos
atributos_numericos = X.select_dtypes(include=np.number).columns.tolist()
atributos_categoricos = X.select_dtypes(exclude=np.number).columns.tolist()

print('Atributos numéricos:', atributos_numericos)
print('Atributos categóricos:', atributos_categoricos)
print('Total de atributos usados:', X.shape[1])

Foram removidas as colunas `streams`, `sucesso`, `track_name` e `artist(s)_name`. A variável `streams` foi retirada porque foi usada para construir o alvo e sua presença no treinamento causaria vazamento de dados. As colunas de identificação textual foram removidas para manter o escopo do MVP focado em atributos estruturados.

In [ ]:
preprocessamento = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), atributos_numericos),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), atributos_categoricos)
    ]
)

## 7. Divisão dos dados

Como o problema é supervisionado, a base foi separada em treino e teste. A divisão foi estratificada pela variável-alvo para preservar a proporção das classes nos dois conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)
print('\nDistribuição no treino:')
print(y_train.value_counts(normalize=True))
print('\nDistribuição no teste:')
print(y_test.value_counts(normalize=True))

## 8. Modelagem e treinamento

Foram avaliados três modelos:

1. **DummyClassifier** como baseline simples;
2. **Random Forest**, por ser um modelo robusto para dados tabulares e capaz de capturar relações não lineares;
3. **Gradient Boosting**, por combinar árvores sequenciais e geralmente apresentar bom desempenho em problemas tabulares.

O baseline serve como referência mínima. Os demais modelos só são úteis se apresentarem desempenho superior a essa regra simples.

In [ ]:
modelos = {
    'Baseline - Dummy': DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

resultados = []

for nome, modelo in modelos.items():
    pipeline = Pipeline(steps=[
        ('preprocessamento', preprocessamento),
        ('modelo', modelo)
    ])

    pipeline.fit(X_train, y_train)

    y_pred_train = pipeline.predict(X_train)
    y_pred_test = pipeline.predict(X_test)

    resultados.append({
        'modelo': nome,
        'accuracy_treino': accuracy_score(y_train, y_pred_train),
        'accuracy_teste': accuracy_score(y_test, y_pred_test),
        'precision_teste': precision_score(y_test, y_pred_test, zero_division=0),
        'recall_teste': recall_score(y_test, y_pred_test, zero_division=0),
        'f1_teste': f1_score(y_test, y_pred_test, zero_division=0)
    })

resultados_df = pd.DataFrame(resultados).sort_values(by='f1_teste', ascending=False)
resultados_df

A comparação inicial mostra se os modelos candidatos superam o baseline. Para este problema, o F1-score foi escolhido como métrica principal porque combina precisão e recall, sendo útil para avaliar o equilíbrio entre acertos da classe de sucesso e controle de falsos positivos.

## 9. Otimização de hiperparâmetros

Foi aplicado Grid Search ao modelo Random Forest. Os hiperparâmetros escolhidos controlam a quantidade de árvores, a profundidade máxima e o número mínimo de amostras por folha. Esses parâmetros influenciam diretamente a capacidade do modelo de capturar padrões sem se ajustar excessivamente aos dados de treino.

In [ ]:
pipeline_rf = Pipeline(steps=[
    ('preprocessamento', preprocessamento),
    ('modelo', RandomForestClassifier(random_state=RANDOM_STATE))
])

param_grid = {
    'modelo__n_estimators': [100, 200],
    'modelo__max_depth': [None, 5, 10],
    'modelo__min_samples_leaf': [1, 3, 5]
}

grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print('Melhores hiperparâmetros:')
print(grid_rf.best_params_)
print(f'Melhor F1 médio na validação cruzada: {grid_rf.best_score_:.4f}')

A otimização foi realizada apenas com os dados de treino, usando validação cruzada. O conjunto de teste foi mantido separado para avaliar o desempenho final em dados não vistos.

In [ ]:
melhor_rf = grid_rf.best_estimator_

y_pred_rf_treino = melhor_rf.predict(X_train)
y_pred_rf_teste = melhor_rf.predict(X_test)

resultado_rf_otimizado = pd.DataFrame([{
    'modelo': 'Random Forest Otimizado',
    'accuracy_treino': accuracy_score(y_train, y_pred_rf_treino),
    'accuracy_teste': accuracy_score(y_test, y_pred_rf_teste),
    'precision_teste': precision_score(y_test, y_pred_rf_teste, zero_division=0),
    'recall_teste': recall_score(y_test, y_pred_rf_teste, zero_division=0),
    'f1_teste': f1_score(y_test, y_pred_rf_teste, zero_division=0)
}])

comparacao_final = pd.concat([resultados_df, resultado_rf_otimizado], ignore_index=True)
comparacao_final.sort_values(by='f1_teste', ascending=False)

## 10. Avaliação dos resultados

Nesta seção, o melhor modelo é avaliado no conjunto de teste por meio de métricas de classificação e matriz de confusão.

In [ ]:
melhor_modelo_nome = comparacao_final.sort_values(by='f1_teste', ascending=False).iloc[0]['modelo']

if melhor_modelo_nome == 'Random Forest Otimizado':
    melhor_modelo = melhor_rf
else:
    modelo_original = modelos[melhor_modelo_nome]
    melhor_modelo = Pipeline(steps=[
        ('preprocessamento', preprocessamento),
        ('modelo', modelo_original)
    ])
    melhor_modelo.fit(X_train, y_train)

y_pred = melhor_modelo.predict(X_test)

print('Melhor modelo:', melhor_modelo_nome)
print('\nRelatório de classificação:')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não sucesso', 'Sucesso'])
disp.plot(cmap='Blues')
plt.title('Matriz de confusão - melhor modelo')
plt.show()

A matriz de confusão permite observar os acertos e erros do modelo em cada classe. Essa análise é importante porque a acurácia isolada pode esconder padrões de erro, principalmente quando há diferença de desempenho entre as classes.

### Análise de overfitting e underfitting

Para avaliar indícios de overfitting ou underfitting, foram comparados os resultados em treino e teste. Quando a performance no treino é muito superior à do teste, há sinal de possível overfitting. Quando ambas são baixas, pode haver underfitting.

In [ ]:
comparacao_final[['modelo', 'accuracy_treino', 'accuracy_teste', 'f1_teste']].sort_values(by='f1_teste', ascending=False)

Se a diferença entre treino e teste for moderada, o modelo apresenta boa capacidade de generalização. Caso a diferença seja alta, a solução pode estar excessivamente ajustada aos dados de treino e precisaria de regularização, simplificação ou mais dados.

## 11. Importância das variáveis

Quando o melhor modelo for baseado em Random Forest, é possível observar quais atributos tiveram maior contribuição relativa para a classificação.

In [ ]:
def obter_nomes_features(preprocessador):
    nomes = []

    if len(atributos_numericos) > 0:
        nomes.extend(atributos_numericos)

    if len(atributos_categoricos) > 0:
        encoder = preprocessador.named_transformers_['cat'].named_steps['encoder']
        nomes.extend(encoder.get_feature_names_out(atributos_categoricos))

    return np.array(nomes)

modelo_final = melhor_modelo.named_steps['modelo']

if hasattr(modelo_final, 'feature_importances_'):
    nomes_features = obter_nomes_features(melhor_modelo.named_steps['preprocessamento'])
    importancias = modelo_final.feature_importances_

    importancia_df = pd.DataFrame({
        'feature': nomes_features,
        'importancia': importancias
    }).sort_values(by='importancia', ascending=False).head(15)

    sns.barplot(data=importancia_df, x='importancia', y='feature')
    plt.title('Top 15 variáveis mais importantes')
    plt.xlabel('Importância')
    plt.ylabel('Variável')
    plt.show()

    display(importancia_df)
else:
    print('O modelo selecionado não possui atributo feature_importances_.')

A análise de importância das variáveis ajuda a interpretar o comportamento do modelo. Espera-se que atributos relacionados à exposição, como presença em playlists e charts, apareçam entre os mais relevantes, pois a análise exploratória já indicava associação entre visibilidade e sucesso.

## 12. Respostas ao checklist do MVP

### Definição do problema

O problema escolhido foi classificar músicas entre maior e menor sucesso, usando como critério a mediana de streams. O objetivo do modelo é prever se uma música pertence ao grupo de maior sucesso da base. Trata-se de um problema de classificação supervisionada, pois existe uma variável-alvo definida. A principal hipótese é que variáveis de exposição, como presença em playlists e charts, ajudam a explicar o desempenho das faixas.

### Descrição dos dados

O dataset utilizado foi o *Most Streamed Spotify Songs 2023*, obtido em repositório público. A base contém informações sobre músicas, artistas, data de lançamento, presença em plataformas e características musicais. A variável-alvo foi criada a partir de `streams`. Uma limitação importante é que a base representa músicas já populares, não todo o catálogo do Spotify.

### Preparação dos dados

Foram tratados valores ausentes por imputação dentro de pipeline. A variável `streams` foi removida dos atributos preditores para evitar vazamento de dados, pois foi usada na criação do alvo. Variáveis numéricas foram padronizadas e variáveis categóricas foram codificadas por One-Hot Encoding. As transformações foram aplicadas de forma adequada à divisão treino/teste por meio de pipelines.

### Divisão dos dados

Foi utilizada divisão treino/teste com 25% dos dados para teste e estratificação pela variável-alvo. A estratégia é adequada porque se trata de um problema supervisionado de classificação e preserva a proporção das classes nos dois conjuntos.

### Modelagem

Foi utilizado um baseline com `DummyClassifier`, além de Random Forest e Gradient Boosting. Os modelos foram comparados por métricas no conjunto de teste. A comparação com treino também permitiu observar possíveis sinais de overfitting ou underfitting.

### Otimização

Foi aplicado Grid Search ao Random Forest, ajustando número de árvores, profundidade máxima e mínimo de amostras por folha. A seleção foi feita com validação cruzada usando F1-score. O teste foi preservado para avaliação final.

### Avaliação

Foram utilizadas acurácia, precisão, recall, F1-score e matriz de confusão. O F1-score foi usado como métrica principal por equilibrar precisão e recall. Os resultados foram analisados em dados não vistos e comparados entre os modelos.

### Conclusão

A melhor solução foi escolhida com base no desempenho no conjunto de teste, especialmente pelo F1-score. O MVP cumpriu o objetivo de construir uma solução de classificação reproduzível, documentada e tecnicamente justificada. Como próximos passos, seria possível testar novas features, usar validação mais robusta, avaliar outros algoritmos e ampliar a base de dados.

## 13. Conclusão

Este MVP teve como objetivo construir uma solução de Machine Learning para classificar músicas do dataset *Most Streamed Spotify Songs 2023* entre faixas de maior e menor sucesso, usando como referência a mediana do número de streams.

Ao longo do trabalho, foram realizadas etapas de análise exploratória, criação da variável-alvo, preparação dos dados, divisão treino/teste, construção de pipelines, treinamento de modelos, otimização de hiperparâmetros e avaliação dos resultados.

Os modelos avaliados foram um baseline simples, Random Forest e Gradient Boosting. A comparação permitiu verificar se modelos mais elaborados trouxeram ganho em relação à regra ingênua. A análise das métricas e da matriz de confusão ajudou a entender o desempenho do melhor modelo em dados não vistos.

A principal limitação do MVP é que a variável-alvo foi criada a partir da própria distribuição de streams da base, e não a partir de uma definição externa de sucesso. Além disso, o dataset contém apenas músicas já relevantes no Spotify, o que limita a generalização para músicas fora desse universo.

Como próximos passos, seria interessante testar outros algoritmos, explorar mais engenharia de atributos, incluir dados de redes sociais ou campanhas de divulgação e avaliar se o modelo mantém desempenho em músicas lançadas em períodos posteriores.